# Phase 1 — Lesson 4
# Time Series Components: Level, Trend, Seasonality, and Noise
## EUR/USD H4 Practice Notebook

Default EUR/USD project dataset: `Data/EURUSD_H4.csv`

### Objectives
- Understand Level, Trend, Seasonality, and Noise.
- Identify them visually in EUR/USD H4 data.
- Distinguish Trend vs Seasonality and Seasonality vs Noise.
- Practice using the real EUR/USD dataset.

> This lesson does not use smoothing, rolling averages, decomposition, stationarity, or forecasting models.

## 1. The four components

$$
\text{Time Series} = \text{Level} + \text{Trend} + \text{Seasonality} + \text{Noise}
$$

A common symbolic form is:

$$
Y_t = L_t + T_t + S_t + \varepsilon_t
$$

- **Level**: typical value around which the series moves.
- **Trend**: general long-term direction.
- **Seasonality**: pattern that repeats at a regular interval.
- **Noise**: irregular movement that does not repeat predictably.

## 2. Import libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

## 3. Load the default EUR/USD H4 dataset

In [ ]:
eurusd = pd.read_csv(
    "Data/EURUSD_H4.csv",
    parse_dates=["DateTime"],
    index_col="DateTime"
)

eurusd = eurusd.sort_index()

eurusd.head()

## 4. Quick data check

In [ ]:
print(f"First timestamp: {eurusd.index.min()}")
print(f"Last timestamp : {eurusd.index.max()}")
print(f"Rows           : {len(eurusd):,}")
print(f"Index sorted   : {eurusd.index.is_monotonic_increasing}")
print(f"Duplicates     : {eurusd.index.duplicated().sum()}")

# Part 1 — Level

The **level** is the typical value around which the series fluctuates during a selected period.

For a simple beginner estimate, we can use the mean Close for one month.

## 5. Level example — January 2025

In [ ]:
january_2025 = eurusd.loc["2025-01"]

level = january_2025["Close"].mean()

print(f"Approximate January 2025 level: {level:.5f}")

In [ ]:
plt.figure(figsize=(15, 6))

plt.plot(january_2025.index, january_2025["Close"], label="Close")
plt.axhline(y=level, linestyle="--", label="Approximate Level")

plt.xlabel("Date")
plt.ylabel("EUR/USD")
plt.title("EUR/USD H4 — Level Example — January 2025")
plt.xticks(rotation=45)
plt.legend()
plt.grid()
plt.show()

### What to look for

- Does the price spend much of the month near the horizontal line?
- Does it move above and below that value?

The line gives a simple visual reference for the **level**.

# Part 2 — Trend

A **trend** is the general direction of a series over time.

- **Uptrend**: generally moves higher.
- **Downtrend**: generally moves lower.
- **No obvious trend**: no clear overall direction.

Trend does **not** mean every observation moves in the same direction.

## 6. Visual trend example — 2025

In [ ]:
eurusd_2025 = eurusd.loc["2025"]

plt.figure(figsize=(15, 6))
plt.plot(eurusd_2025.index, eurusd_2025["Close"])
plt.xlabel("Date")
plt.ylabel("EUR/USD")
plt.title("EUR/USD H4 Closing Price — 2025")
plt.xticks(rotation=45)
plt.grid()
plt.show()

### Ask yourself

1. Is the general direction upward?
2. Is it downward?
3. Or is there no obvious overall direction?

Focus on the big picture rather than every H4 movement.

## 7. Beginning vs end — descriptive check

In [ ]:
first_close = eurusd_2025["Close"].iloc[0]
last_close = eurusd_2025["Close"].iloc[-1]

print(f"First Close in 2025: {first_close:.5f}")
print(f"Last Close in 2025 : {last_close:.5f}")

if last_close > first_close:
    print("End value is above the beginning value.")
elif last_close < first_close:
    print("End value is below the beginning value.")
else:
    print("Beginning and ending values are equal.")

> Comparing first and last values is only a supporting descriptive check. Always inspect the full chart before describing a trend.

# Part 3 — Seasonality

**Seasonality** is a pattern that repeats at a regular interval.

For EUR/USD H4 data, the price itself may not show a simple repeating daily pattern. A useful beginner exploration is to examine whether **market activity** changes repeatedly by H4 candle hour.

We will use the candle High-Low range as a simple activity measure.

## 8. Create candle range

$$
\text{Range} = \text{High} - \text{Low}
$$

In [ ]:
eurusd["Range"] = eurusd["High"] - eurusd["Low"]

eurusd[["High", "Low", "Range"]].head()

## 9. Average H4 range by candle hour

In [ ]:
hourly_range = eurusd.groupby(eurusd.index.hour)["Range"].mean()

hourly_range

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(hourly_range.index, hourly_range.values)
plt.xlabel("Candle Start Hour")
plt.ylabel("Average High-Low Range")
plt.title("EUR/USD H4 — Average Range by Hour")
plt.xticks(hourly_range.index)
plt.grid(axis="y")
plt.show()

### Interpretation

If some H4 hours repeatedly have larger average ranges than others, there may be a recurring **intraday activity pattern**.

That is different from saying the price itself repeats every day.

## 10. Average range by weekday

In [ ]:
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]

day_range = eurusd.groupby(eurusd.index.day_name())["Range"].mean()
day_range = day_range.reindex(day_order)

day_range

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(day_range.index, day_range.values)
plt.xlabel("Day")
plt.ylabel("Average High-Low Range")
plt.title("EUR/USD H4 — Average Range by Day of Week")
plt.xticks(rotation=45)
plt.grid(axis="y")
plt.show()

> A historical seasonal pattern does not guarantee that the pattern will continue in the future. Financial-market behavior changes over time.

# Part 4 — Noise

**Noise** is irregular short-term movement that does not form a clear repeating pattern.

Financial price series naturally contain a lot of short-term irregular movement.

## 11. Visual noise example — one trading week

In [ ]:
eurusd_week = eurusd.loc["2025-01-06":"2025-01-10"]

plt.figure(figsize=(15, 6))
plt.plot(eurusd_week.index, eurusd_week["Close"], marker="o")
plt.xlabel("Date")
plt.ylabel("EUR/USD")
plt.title("EUR/USD H4 — Short-Term Irregular Movement")
plt.xticks(rotation=45)
plt.grid()
plt.show()

### What to look for

Short-term movements may go up, down, then reverse again. These irregular fluctuations are part of what we call **noise**.

Do not assume every small movement is a meaningful trend.

# Trend vs Seasonality

| Trend | Seasonality |
|---|---|
| General long-term direction | Repeating pattern |
| Does not need to repeat | Repeats at a regular interval |
| Example: generally rising for months | Example: activity repeatedly higher at certain hours |

# Seasonality vs Noise

| Seasonality | Noise |
|---|---|
| Repeats | Irregular |
| Has a time-based pattern | No reliable repeating pattern |
| Linked to a regular interval | Random-looking short-term variation |

# 12. Four questions to ask

### Level
Around what value does the series usually move?

### Trend
Is the general direction up, down, or unclear?

### Seasonality
Is there a pattern that repeats at a regular interval?

### Noise
Which movements appear irregular and non-repeating?

# Practice Exercises

## Exercise 1 — Level

Use **March 2025**.

1. Filter March 2025.
2. Calculate mean Close.
3. Plot Close.
4. Add a horizontal line for the approximate level.

In [ ]:
# Write your Exercise 1 code here


## Exercise 2 — Trend

Plot EUR/USD Close for **2024** and classify it visually as:

- Uptrend
- Downtrend
- No obvious trend

In [ ]:
# Write your Exercise 2 code here


## Exercise 3 — Seasonality

Calculate average `Range` by H4 candle hour.

Answer:
1. Which hour has the largest average range?
2. Which hour has the smallest average range?
3. Is activity identical across all H4 periods?

In [ ]:
# Write your Exercise 3 code here


## Exercise 4 — Noise

Choose one trading week in 2025, plot Close with markers, and describe the irregular short-term movement you observe.

In [ ]:
# Write your Exercise 4 code here


# Final Practice Task

Choose **one month in 2025** and perform a short component analysis.

### Level
- Calculate average Close.
- Plot Close with a horizontal level line.

### Trend
State: Uptrend, Downtrend, or No obvious trend.

### Seasonality
Calculate average `Range` by H4 candle hour.

### Noise
Identify irregular short-term movement.

Finally write one conclusion for each component.

In [ ]:
# Final Practice Task

# Choose one month in 2025
# Example:
# practice_month = eurusd.loc["2025-03"]

practice_month = eurusd.loc["2025-__"]

# 1. LEVEL


# 2. TREND


# 3. SEASONALITY


# 4. NOISE


## Final conclusions

**Level:**  

**Trend:**  

**Seasonality:**  

**Noise:**  

# Lesson 4 Cheat Sheet

## Level
Typical value around which the series moves.

```python
level = data["Close"].mean()
```

## Trend
General direction over time:
- Uptrend
- Downtrend
- No obvious trend

## Seasonality
A pattern that repeats at a regular interval.

```python
eurusd["Range"] = eurusd["High"] - eurusd["Low"]
eurusd.groupby(eurusd.index.hour)["Range"].mean()
```

## Noise
Irregular short-term movement that does not repeat predictably.

## Main equation

$$
\text{Time Series} = \text{Level} + \text{Trend} + \text{Seasonality} + \text{Noise}
$$